# Job 50099945 — the first L\*(T) depth ladder measured *without* the adjacency-order leak, at N=48

Every L\* in this project before today (findings 14, 22, 25, 26, 31, 32) was measured on the
leaked encoding, where a model can read part of the colouring off the order of the adjacency
rows (finding 16). This run is the first rung measured with `--shuffle-adj`, so the shortcut is
gone and the depth question is about reasoning rather than about row order.

**Headline (finding 33).** At N=48, leak-free, ff=1024, and the 240000 x 100ep budget that
solved N=24: **L\*(48) > 6**. Six layers is the top of the ladder and it does *not* clear the
0.90 gate at density 0.60 in any depth bin, so L\* is **undefined here, not large** — the next
series goes L=8 and L=12 *upward*, not L=3 downward. On the n=1500 shared pool L=6 scores
**0.6153 [0.5904, 0.6396]** against 0.3153 at L=4 and 0.0093 at L=3.

**Why that is the number to quote.** The *leaked* ladder puts L\*(48) at **4** (finding 14).
Leak-free needs more than 6 at three times the data, so the adjacency-order leak was worth at
least **two rungs of depth** at this N.

Deck page: `notebooks/sudoku_n96_rungs_2026-10-29/page9.png` (page 9 of the rungs deck).


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S = os.environ["SCRATCH"] + "/spin_gnn"
LF48 = f"{S}/sudoku_n48_leakfree_lstar_2026-10-29"   # 50099945, this run
JOB = "50099945"
GATE = 0.90          # the solve-rate threshold that defines L*(T)
pd.set_option("display.width", 220)

LAB = {"d0256_h04_L06_ff01024_lr1e-03_pre_shuf": "L=6",
       "d0256_h04_L04_ff01024_lr1e-03_pre_shuf": "L=4",
       "d0256_h04_L03_ff01024_lr1e-03_pre_shuf": "L=3"}


def with_L(path):
    t = pd.read_csv(path)
    t["L"] = t.checkpoint.str.extract(r"_L(\d+)_").astype(int)
    return t


bins = with_L(f"{LF48}/depth_lstar_{JOB}.csv")
pool = with_L(f"{LF48}/sharedpool_n1500_{JOB}.csv")
print(sorted(bins.checkpoint.unique()))


## 1. Metric of record: the shared n=1500 pool

`sudoku_agg_eval` builds one pool at `seed_start=50000000000`, far above every training seed, and
instance *i* is drawn at `seed_start + i`, so the three cells see **byte-identical prompts** and
every delta below is paired instance by instance (McNemar exact, two-sided). All three cells have
`shuffle_adj=True`, `hint_mode=removal`, N=48, degree 5, q=4, and differ **only** in
`num_layers`. The pool's own reference is L=6, the top of the ladder.

`outcome_accuracy` 0.996-1.000 and `truncated_rate` 0.000 everywhere: when these models answer
they answer in the right format and inside the token budget, so the solve-rate gap is reasoning,
not truncation or malformed output.


In [ ]:
cols = ["checkpoint", "n_instances", "solve_rate", "wilson_lo", "wilson_hi",
        "delta", "discordant_won", "discordant_lost", "mcnemar_exact_p",
        "truncated_rate", "outcome_accuracy"]
t = pool.copy()
t["checkpoint"] = t.checkpoint.map(LAB)
print("=== shared pool, n=1500, reference = L=6 (delta and p are vs that reference)")
print(t[cols].round(4).to_string(index=False))


## 2. The depth gate: solve rate per DD-round bin

`T` is the number of Direct-Deduction rounds the instance actually needs, computed by the solver,
so a row is "instances requiring exactly T rounds". **L\*(T) is the smallest ladder rung whose
solve rate on that bin is >= 0.90.** Two densities were scored: 0.75 (where the realized density
matches the request to six decimals in every bin) and **0.60, the density the pre-registration
named**.

The two densities answer different questions here, and only one of them is answerable:

* **rho = 0.75** — L=6 clears the gate at T=2 and T=3 and misses at T=4, so **L\*(2) = L\*(3) = 6
  and L\*(4) > 6**: the gate crosses between three and four rounds.
* **rho = 0.60** — L=6 clears **0 of 6** bins, L=4 and L=3 clear none. With no cell above the gate
  there is no rung to compare, which is exactly the failure mode this series was ordered
  deepest-cell-first to catch.


In [ ]:
for rho in (0.75, 0.60):
    sub = bins[bins.rho_requested == rho]
    piv = sub.pivot_table(index="T", columns="L", values="solve_rate")
    n = sub.pivot_table(index="T", columns="L", values="n")[sub.L.max()].astype(int)
    piv = piv[sorted(piv.columns, reverse=True)]
    piv.columns = [f"L={c}" for c in piv.columns]
    piv.insert(0, "n", n)
    rr = sub.realized_rho
    print(f"=== rho requested {rho:.2f}, REALIZED {rr.min():.6f}-{rr.max():.6f} in every bin")
    print(piv.round(4).to_string())
    clears = {L: int(((sub.L == L) & (sub.solve_rate >= GATE)).sum()) for L in (6, 4, 3)}
    print("bins at or above the 0.90 gate:", clears, f"of {sub['T'].nunique()}\n")


## 3. The pre-registrations, scored

Written into `scripts/slurm/sudoku2_leakfree_lstar.sbatch` (lines 64-86) **before** the run:

| | claim | verdict |
|---|---|---|
| (a) | L\*(T) rises with T by at least one rung at rho=0.60 | **CONFIRMED at 0.75, UNTESTABLE at 0.60** |
| (b) | leak-free L\* exceeds the leaked L\* | **CONFIRMED** (leaked 4 vs leak-free > 6) |
| (c) | the ff=256 column behaves differently | **not tested** — this ladder is at fixed ff=1024 |
| (d) | deep cells' per-T curves are flatter than shallow cells' | **FALSIFIED as stated, confounded as measured** |

(d) is the interesting failure. L=6 falls 0.517 over T=3-8 while L=4 falls 0.227 — the opposite of
the prediction — but L=4 sits at 0.28 and L=3 at 0.00, and a cell near the floor **cannot** fall.
The comparison needs two cells that are both well above the floor, which this ladder does not
contain at rho=0.60.


In [ ]:
# Cochran-Armitage trend z over ordered bins. Negative = falling with x.
def ca_trend(ks, ns, xs):
    ks, ns, xs = np.asarray(ks, float), np.asarray(ns, float), np.asarray(xs, float)
    N, K = ns.sum(), ks.sum()
    pbar, xbar = K / N, (ns * xs).sum() / N
    var = pbar * (1 - pbar) * (ns * (xs - xbar) ** 2).sum()
    return (ks * (xs - xbar)).sum() / np.sqrt(var) if var > 0 else float("nan")


for rho in (0.75, 0.60):
    for L in (6, 4):
        r = bins[(bins.rho_requested == rho) & (bins.L == L)].sort_values("T")
        z = ca_trend((r.solve_rate * r.n).round(), r.n, r["T"])
        print(f"rho={rho:.2f} L={L}: {r.solve_rate.max():.3f} -> {r.solve_rate.min():.3f} "
              f"over T={int(r['T'].min())}-{int(r['T'].max())}, "
              f"drop {r.solve_rate.max() - r.solve_rate.min():.3f}, Cochran-Armitage z = {z:.1f}")


## 4. Comparison to the previous best on the same eval

**There is no instance-paired comparison available, and saying so is part of the result.** The
n=1500 pool is built from the task's own `(N, degree, q, shuffle_adj)`, so an N=48 leak-free pool
shares no instances with the N=24 leak-free pool (findings 18/27) or with the *leaked* N=48 ladder
(finding 14). The comparisons below are therefore between **L\*** values and between solve rates
on different pools, not paired tests:

| run | N | leak | best cell | result |
|---|---|---|---|---|
| finding 14 | 48 | leaked | L=4 / ff=1024 | **L\*(48) = 4** |
| finding 18 | 24 | leak-free | L=12 / ff=1024 | solved: 1.0000 on its own pool |
| **this run** | **48** | **leak-free** | **L=6 / ff=1024** | **L\*(48) > 6**, pool 0.6153 |

The paired contrasts that *are* valid are the three inside this run's own pool (section 1): L=6 vs
L=4 is +0.300 (36-486 discordant, p = 8.4e-102) and L=6 vs L=3 is +0.606 (0-909, p < 1e-200).
Depth is doing something very large here; what is not established is where it stops.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=False)
STY = {6: ("#2b6cb0", "o", "L=6 (top of the ladder)"), 4: ("#b7791f", "s", "L=4"),
       3: ("#c53030", "^", "L=3")}
for ax, rho in zip(axes, (0.75, 0.60)):
    sub = bins[bins.rho_requested == rho]
    for L, (col, mk, lab) in STY.items():
        r = sub[sub.L == L].sort_values("T")
        ax.plot(r["T"], r.solve_rate, mk + "-", color=col, lw=2.5, ms=8, label=lab)
    ax.axhline(GATE, color="k", ls="--", lw=2)
    ax.set_ylim(-0.05, 1.08)
    ax.set_xticks(sorted(sub["T"].unique()))
    ax.set_ylabel("solve rate")
    ax.set_title(f"density {rho:.2f} (dashed = the 0.90 gate that defines L*)")
axes[0].legend(loc="lower left")
axes[1].set_xlabel("T, the number of DD rounds the instance needs")
fig.tight_layout()


## 5. Provenance, config, limitations

* **Job** `50099945`, submitted 2026-10-29 01:45:23, COMPLETED 2026-10-29 13:47:56
  (`sacct` Elapsed 09:50 against `scontrol` RunTime 05:44 — consistent with a requeue; the run
  completed and its outputs are intact).
* **Commit at submit time** `bce0923`. Script `scripts/slurm/sudoku2_leakfree_lstar.sbatch`,
  invoked with `NNODES=48 LAYERS="6 4 3" NTRAIN=240000 EPOCHS=100`.
* **Config, identical across the three cells except `num_layers`:** N=48, degree 5, q=4,
  rho in [0.3, 0.95] for training, removal hints, `--shuffle-adj`, d_model=256, nhead=4,
  ff=1024, pre-norm, lr=1e-3 constant, batch=32, **seed=0**.
* **Outputs** `$SCRATCH/spin_gnn/sudoku_n48_leakfree_lstar_2026-10-29/`:
  `depth_lstar_50099945.csv`, `sharedpool_n1500_50099945.csv`,
  `sharedpool_n1500_50099945_per_instance.npz`, and the `L03/ L04/ L06/` checkpoints.
  On `$SCRATCH`, so subject to the 90-day purge — nothing here is a final model, and the CSVs are
  small enough to re-derive from the checkpoints while they last.

**Limitations.** One seed per cell, one checkpoint per cell, three rungs. The rho=0.75 reading
rests on three bins of 674 / 531 / 194 instances. The gate is a threshold on a continuous curve,
so L\* moves if the gate does: the T=4 bin at rho=0.75 sits at 0.7835, so any gate above 0.79
leaves L\*(4) > 6, and at rho=0.60 L=6's best bin is 0.8310, so the "L=6 clears nothing" verdict
survives every gate above 0.84. **L\*(48, leak-free, ff=1024) lies in (6, 12]**:
twelve layers solved N=24 outright, and nothing here bounds it from above at N=48.

**Next.** L=8 and L=12 at ff=1024 on this same ladder, same budget, same bins — about 5 h per cell
on `sganguli`, and each needs its own pre-registration before submission.
